[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_6_8/04_tag_6_8_filter_visualisieren_featuremaps.ipynb)

# Tag 6-8 - 04 Filter visualisieren und Feature Maps lesen

Filter werden zuerst auf einer Ziffer mit Label und danach auf einem größeren Graustufenbild angewendet. Die Feature Maps bleiben bewusst graustufig.


In [ ]:
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = False

from sklearn.datasets import load_digits
from PIL import Image
from skimage import data, color, transform


## Ziffer mit Label


In [ ]:
digits = load_digits()
idx = 42
digit = digits.images[idx].astype(np.float32) / 16.0
label = digits.target[idx]
digit_large = np.array(Image.fromarray((digit * 255).astype(np.uint8)).resize((96, 96), resample=Image.Resampling.BICUBIC)) / 255.0

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(digit, cmap="gray", vmin=0, vmax=1)
axes[0].set_title(f"8x8, Label {label}")
axes[1].imshow(digit_large, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("vergrößert")
for ax in axes:
    ax.axis("off")


## Größeres Graustufenbild

`skimage.data.camera()` ist ein paketlokales Beispielbild, kein Download.


In [ ]:
camera = data.camera().astype(np.float32) / 255.0
camera_small = transform.resize(camera, (160, 160), anti_aliasing=True).astype(np.float32)
plt.imshow(camera_small, cmap="gray", vmin=0, vmax=1)
plt.title("größeres Graustufenbild: camera")
plt.axis("off");


## Filter und Feature Maps


In [ ]:
kernels = {
    "vertikale Kante": np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=np.float32),
    "horizontale Kante": np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=np.float32),
    "weichzeichnen": np.ones((3, 3), dtype=np.float32) / 9.0,
    "schärfen": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32),
}

def convolve2d(image, kernel, padding=1, stride=1):
    padded = np.pad(image, padding, mode="edge")
    out_h = (padded.shape[0] - kernel.shape[0]) // stride + 1
    out_w = (padded.shape[1] - kernel.shape[1]) // stride + 1
    output = np.zeros((out_h, out_w), dtype=np.float32)
    for row in range(out_h):
        for col in range(out_w):
            r = row * stride
            c = col * stride
            patch = padded[r:r + kernel.shape[0], c:c + kernel.shape[1]]
            output[row, col] = np.sum(patch * kernel)
    return output

fig, axes = plt.subplots(2, len(kernels), figsize=(11, 5))
for col, (name, kernel) in enumerate(kernels.items()):
    fmap = convolve2d(camera_small, kernel, padding=1)
    axes[0, col].imshow(kernel, cmap="gray")
    axes[0, col].set_title(name)
    axes[1, col].imshow(fmap, cmap="gray")
    axes[1, col].set_title("Feature Map")
    for ax in axes[:, col]:
        ax.axis("off")
plt.tight_layout()
